# ZooKeeper historical baseline in a disposable QEMU guest

A CI companion to D12 and T12a: execute the repository's ZooKeeper install/start/stop playbooks and verify real data. This is a single-node baseline, not an ensemble or a migration test yet. The guest disk must survive every phase; never format or recreate it between phases.


In [ ]:
import os
from pathlib import Path
import subprocess
from kazoo.client import KazooClient

root = Path.cwd()
state = Path(os.environ['CI_STATE'])
inventory = state / 'inventory.yml'
variables = root / 'ci/zookeeper-vars.yml'

def run(*args):
    # Unlike !ansible, a nonzero exit must fail the notebook and CI.
    print('+', ' '.join(map(str, args)), flush=True)
    subprocess.run(list(map(str, args)), check=True, timeout=900)

def playbook(path):
    run('ansible-playbook', '-i', inventory, '-e', '@' + str(variables), root / path)

def read_data():
    zk = KazooClient(hosts='127.0.0.1:22181', timeout=15)
    try:
        zk.start(timeout=60)
        return zk.get('/notebook-ci/persistent')[0]
    finally:
        zk.stop()
        zk.close()


## Bind to the guest, then install the historical baseline

In [ ]:
run('ansible', '-i', inventory, 'all', '-m', 'ping')
playbook('playbooks/install_zookeeper.yml')
playbook('playbooks/start_zookeeper-server.yml')
playbook('ci/enable-zookeeper.yml')


In [ ]:
run('ansible', '-i', inventory, 'all', '-b', '-a', 'systemctl is-active zookeeper-server')
run('ansible', '-i', inventory, 'all', '-b', '-a', '/opt/zookeeper/current/bin/zkServer.sh --config /etc/zookeeper/conf status')


## Write persistent data and verify the actual server version

In [ ]:
zk = KazooClient(hosts='127.0.0.1:22181', timeout=15)
expected = b'created-by-zookeeper-3.6.2-before-restart'
try:
    zk.start(timeout=60)
    version = zk.server_version()
    print('Server version:', version)
    assert version == (3, 6, 2), version
    zk.ensure_path('/notebook-ci')
    zk.create('/notebook-ci/persistent', expected, ephemeral=False)
    assert zk.get('/notebook-ci/persistent')[0] == expected
finally:
    zk.stop()
    zk.close()


## Stop/start the service and read the same data

In [ ]:
playbook('playbooks/stop_zookeeper-server.yml')
playbook('playbooks/start_zookeeper-server.yml')
assert read_data() == expected
print('Persistent data survived service stop/start')


## Reboot the same guest and verify persistent data again

In [ ]:
run('ansible', '-i', inventory, 'all', '-b', '-m', 'reboot', '-a', 'reboot_timeout=180')
assert read_data() == expected
print('Persistent data survived guest reboot')


## Next phase: migration

A future scenario will stop this baseline, install a separately pinned target release on the **same data directory**, start it, and verify both old and newly written data. Hadoop migration will also require HDFS data checks and YARN jobs, with its supported upgrade/finalize sequence. This baseline does not establish migration compatibility.
